In [1]:
from typing import cast
from datasets import DatasetDict, load_dataset
import gymnasium as gym
from tinkerbell.types import ModelInput, TensorData
from pydantic import BaseModel
from tinkerbell.types import ModalDeployConfig
from tinkerbell.client import ServiceClient
from tinkerbell.renderer import Renderer

### Inference client

In [2]:
MODEL_ID = "Qwen/Qwen3-0.6B"
GPU_TYPE = "A100"
NUM_GPUS = 1

deploy_config = ModalDeployConfig(
    gpu=GPU_TYPE, 
    num_gpus=NUM_GPUS,
    timeout=86400,
    container_idle_timeout=600,
    max_inputs=200,
    max_wait_time=1200.0,
)


In [3]:
service_client = ServiceClient.deploy_or_connect(deploy_config)

sampling_client = service_client.create_sampling_client(
    model_id=MODEL_ID,
    tp_size=NUM_GPUS,
    model_name="qwen3-06b",
)

/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/tinkerbell/types/deploy.py:40: DeprecationError: 2025-02-11: Looking up class methods using Function.from_name will be deprecated in a future version of Modal.
Use modal.Cls.from_name instead, e.g.

deploy_on_modal = modal.Cls.from_name("tinkerbell-service", "deploy_on_modal")
instance = deploy_on_modal(...)
instance.<locals>.serve.remote(...)

  existing_function = modal.Function.from_name(
W1209 20:00:28.289000 40465 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/tinkerbell/service/server.py:675: DeprecationError: 2025-02-24: We have renamed several parameters related to autoscaling. Please update your code to use the following new names:

- container_idle_timeout -> scaledown_window

See https://modal.com/docs/guide/modal-1-0-migration for more details.
  @app.function(

Starting server deployment...
Deploying server on Modal...
⠸ Creating objects.....
⠦ Creating objects...honPackage:tinkerbell: Uploaded 2/31 files
⠏ Creating objects...honPackage:tinkerbell: Finalizing index of 31 files
├── 🔨 Created mount PythonPackage:tinkerbell
⠙ Creating objects...n_modal.<locals>.serve...
├── 🔨 Created mount PythonPackage:tinkerbell
└── 🔨 Created web function deploy_on_modal.<locals>.serve => 
    https://jesterlabs--training-service.modal.run
✓ Created objects.
├── 🔨 Created mount PythonPackage:tinkerbell
└── 🔨 Created web function deploy_on_modal.<locals>.serve => 
    https://jesterlabs--training-service.modal.run
✓ App deployed in 1.374s! 🎉

View Deployment: 
https://modal.com/apps/jesterlabs/main/deployed/tinkerbell-service
Deployed new server at: https://jesterlabs--training-service.modal.run


/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/tinkerbell/service/server.py:700: DeprecationError: 2025-02-11: Looking up class methods using Function.from_name will be deprecated in a future version of Modal.
Use modal.Cls.from_name instead, e.g.

deploy_on_modal = modal.Cls.from_name("tinkerbell-service", "deploy_on_modal")
instance = deploy_on_modal(...)
instance.<locals>.serve.remote(...)

  return modal.Function.from_name(


In [4]:
sampling_client.wait_until_ready()

In [7]:
# Full model: General knowledge (uses all parameters)
full_model_conversations = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me about machine learning."},
    ],
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me about bayesian inference."},
    ],
]

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
renderer = Renderer(tokenizer)

In [9]:
input_data = renderer.build_chat_samples(
    messages=full_model_conversations,
    mask_value=-100,
)


In [10]:
input_data[0].model_input.input_ids

TensorData(data=[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 40451, 752, 911, 5662, 6832, 13, 151645, 198], dtype='int64', shape=[22])

In [11]:
sampling_params = {
    "max_new_tokens": 1024,
    "temperature": 0.7,
    "top_p": 0.9,
    # "top_k": 50,
    # "return_text_in_logprobs": False
}


In [12]:
input_data

[Datum(model_input=ModelInput(input_ids=TensorData(data=[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 40451, 752, 911, 5662, 6832, 13, 151645, 198], dtype='int64', shape=[22]), attention_mask=TensorData(data=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype='int64', shape=[22]), additional_inputs=None), loss_fn_inputs={'labels': TensorData(data=[8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 40451, 752, 911, 5662, 6832, 13, 151645, 198, -100], dtype='int64', shape=[22])}),
 Datum(model_input=ModelInput(input_ids=TensorData(data=[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 40451, 752, 911, 22708, 29221, 44378, 13, 151645, 198], dtype='int64', shape=[23]), attention_mask=TensorData(data=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype='int64', shape=[23]), additional_inputs=None), loss_fn_inputs={'labels': TensorData(data=[8948, 1

In [13]:
sample_futures = sampling_client.sample(
    input_ids=input_data[0].model_input.input_ids,
    sampling_params=sampling_params,
    return_logprob=True,
    top_logprobs_num=256,
)

OSError: qwen3-06b is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [14]:
out = sample_futures.result()

In [15]:
print(out.outputs[0])


<think>
Okay, the user asked about machine learning. I need to explain it clearly. Let me start by defining it. Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. 

I should mention key concepts like algorithms, data, and models. Then, explain different types: supervised, unsupervised, reinforcement. Maybe give examples like image recognition or recommendation systems. 

Wait, should I include some real-world applications? Like healthcare or finance? That would make it more relatable. Also, talk about how it's used in big data and automation. 

Make sure to highlight that ML doesn't need human input, which is a big point. Maybe mention the benefits like efficiency and accuracy. Avoid jargon but keep it accessible. Check if the explanation covers all main aspects without getting too technical. Alright, that should cover it.
</think>

Machine learning (ML) is a subset of artificial intelligence (AI) that

In [16]:
print(out.model_dump().keys())

dict_keys(['outputs', 'tokens_generated', 'logprobs', 'top_logprobs', 'output_token_ids', 'finish_reasons', 'meta_info'])


In [17]:
print(out.model_dump()["meta_info"].keys())

dict_keys(['id', 'finish_reason', 'prompt_tokens', 'weight_version', 'total_retractions', 'input_token_logprobs', 'output_token_logprobs', 'input_top_logprobs', 'output_top_logprobs', 'completion_tokens', 'cached_tokens', 'e2e_latency', 'response_sent_to_client_ts'])


In [5]:
output_token_ids

NameError: name 'output_token_ids' is not defined

In [ ]:
def make_logprobs_tensor(logprobs: list[list[list]], vocab_size: int) -> torch.Tensor:
    """
    Convert logprobs from format [[logprob, token_id, None], ...] to a tensor.
    
    Args:
        logprobs: List of lists, where each inner list contains top-k logprobs
                in format [logprob_value, token_id, None]
        vocab_size: Size of the vocabulary
    
    Returns:
        Tensor of shape (vocab_size, num_tokens) containing logprob values
        at the correct token_id positions, with -inf elsewhere
    """
    num_tokens = len(logprobs)
    # Initialize tensor with -inf (logprob of 0 probability)
    result = torch.full((vocab_size, num_tokens), float('-inf'), dtype=torch.float32)

    # Fill in the logprobs at the correct token_id positions
    for token_pos_idx, token_position in enumerate(logprobs):
        for entry in token_position:
            if isinstance(entry, list) and len(entry) >= 2:
                # Extract logprob value (first element) and token_id (second element)
                logprob_value = entry[0]
                token_id = entry[1]
                # Place logprob at the correct position
                if isinstance(token_id, int) and 0 <= token_id < vocab_size:
                    result[token_id, token_pos_idx] = logprob_value
    return result

logprobs_tensor = make_logprobs_tensor(out.model_dump()["meta_info"]["output_top_logprobs"], tokenizer.vocab_size)
print(logprobs_tensor.shape)


torch.Size([151643, 485])


In [28]:
out.model_dump()["meta_info"]["output_top_logprobs"][0][0:10]

[[-0.00015367257583420724, 151644, None],
 [-8.928723335266113, 151667, None],
 [-10.893010139465332, 151668, None],
 [-13.571579933166504, 151645, None],
 [-14.46444034576416, 7336, None],
 [-15.535866737365723, 151665, None],
 [-16.250154495239258, 151658, None],
 [-16.78586769104004, 151657, None],
 [-17.14301109313965, 35224, None],
 [-17.14301109313965, 86383, None]]

In [6]:
output_token_ids = [item[1] for item in out.model_dump()["meta_info"]["output_token_logprobs"]]


NameError: name 'out' is not defined

In [7]:
print(output_token_ids)

NameError: name 'output_token_ids' is not defined

In [ ]:
print(out)

SampleResponse(outputs=['</think>\n\nMachine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and models that enable computers to learn from data, identify patterns, and make decisions with minimal human intervention. The goal is to create systems that improve their performance at tasks over time through experience, without being explicitly programmed for each specific task.\n\n### Key Concepts in Machine Learning\n\n1. **Data**: Machine learning relies heavily on data. The quality and quantity of data significantly influence the effectiveness of the models. Data can be structured (e.g., numerical values in a spreadsheet) or unstructured (e.g., text, images, audio).\n\n2. **Features and Labels**: In supervised learning, data is often labeled, meaning each data point has a known output or "label." Features are the input variables used to predict the label.\n\n3. **Model Training**: The process of feeding data into an algorithm to learn p

In [ ]:
# Extract token IDs from the response
# SGLang returns output_ids at top level, but if parsing didn't extract it,
# we can get it from meta_info or extract from logprobs

# Method 1: Check if output_token_ids is already populated
if out.output_token_ids is not None:
    token_ids = out.output_token_ids[0]  # First sequence
    print(f"Token IDs from output_token_ids: {token_ids}")
else:
    # Method 2: Try to get from meta_info if SGLang stores it there
    if "output_ids" in out.meta_info:
        token_ids = out.meta_info["output_ids"]
        print(f"Token IDs from meta_info['output_ids']: {token_ids}")
    # Method 3: Extract from output_top_logprobs (each position has [logprob, token_id, None])
    # The first token in each position's top logprobs is the generated token
    elif "output_top_logprobs" in out.meta_info and out.meta_info["output_top_logprobs"]:
        top_logprobs = out.meta_info["output_top_logprobs"][0]  # First sequence
        token_ids = [token_entry[1] for token_entry in top_logprobs if isinstance(token_entry, list) and len(token_entry) > 1]
        print(f"Token IDs extracted from output_top_logprobs: {token_ids}")
    # Method 4: Extract from input_token_logprobs for output tokens (if available)
    elif "output_logprobs" in out.meta_info and out.meta_info["output_logprobs"]:
        output_logprobs = out.meta_info["output_logprobs"][0]
        token_ids = [entry[1] if isinstance(entry, list) and len(entry) > 1 else None for entry in output_logprobs]
        token_ids = [tid for tid in token_ids if tid is not None]
        print(f"Token IDs extracted from output_logprobs: {token_ids}")
    else:
        print("Could not find token IDs in response. Check out.meta_info.keys() to see available fields.")
        print(f"Available meta_info keys: {list(out.meta_info.keys())}")

# Store in a variable for easy access
output_token_ids = token_ids if 'token_ids' in locals() else None
print(f"\nFinal token IDs: {output_token_ids}")


In [ ]:
print(json.loads(out.json()).keys())

In [ ]:
print(len(out.meta_info["output_top_logprobs"][0]))

In [ ]:
tokenizer.vocab_size

In [ ]:
log_probs = out.meta_info["output_top_logprobs"]
print(log_probs[0])

In [ ]:
len(log_probs)

In [ ]:
import json
print(json.dumps(out.meta_info["input_token_logprobs"], indent=2))

In [ ]:
print(out)

### GSM8K

In [ ]:
def extract_gsm8k_final_answer(text: str) -> str:
    """Extract the final numeric/string answer from a GSM8K solution field.

    GSM8K format typically places the final answer on a line starting with
    '####'. We take the substring following '####' on the last such line.
    """
    import re
    lines = text.splitlines()
    for line in reversed(lines):
        s = line.strip()
        if s.startswith("####"):
            content = s[4:].strip()
            if content.startswith(":"):
                content = content[1:].strip()
            content = content.replace(",", "").strip()
            return content
    matches = re.findall(r"####\s*(.+)", text)
    if matches:
        return matches[-1].strip()
    raise ValueError("No GSM8K final answer found")

def extract_token_string(text: str, token: str = "think") -> str:
    """Extract everything between <token> and </token> tags.
    
    Returns the content between the first occurrence of <token> and </token>.
    If no match is found, raises ValueError.
    """
    import re
    pattern = rf'<{token}>(.*?)</{token}>'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1)
    raise ValueError("No <token>...</token> tags found")

In [ ]:
test = """
<think>
This is a test thinking...
</think>
<answer>
This is a test answer.
</answer>
"""

In [ ]:
print(extract_token_string(test, "think"))
print(extract_token_string(test, "answer"))

In [ ]:
ds = cast(DatasetDict, load_dataset("openai/gsm8k", name="main"))
train_dataset = ds["train"]

In [ ]:
train_dataset[0]

In [ ]:
extract_gsm8k_final_answer(train_dataset[0]["answer"])

In [ ]:
class Action(BaseModel):
    logprobs: TensorData
    data: TensorData

class Transition(BaseModel):
    state: TensorData
    action: Action
    reward: float
    done: bool